# Downside Risk

## Modern Portfolio Theory and PMPT

A comparison of variance-based risk with downside-sensitive portfolio evaluation.

## §1 Motivation & Derivation

Modern Portfolio Theory (MPT) defines portfolio risk using variance:

$$
\sigma_p^2 = w^\top \Sigma w
$$

where:

- $w$ is the portfolio weight vector, so each entry of $w$ gives the fraction of total wealth invested in one asset,
- $w^\top$ is the transpose of that weight vector,
- $\Sigma$ is the covariance matrix of asset returns,
- $\sigma_p^2$ is the variance of total portfolio returns, i.e. the MPT measure of portfolio risk.

This formula is elegant and computationally convenient, which is why every covariance-estimation module in this project ultimately feeds into an MPT-style optimizer. However, variance penalizes **all dispersion** around the mean, including upside surprises.

That is the core criticism of Post-Modern Portfolio Theory (PMPT): investors do not dislike upside volatility; rather, they dislike returns falling below some target or minimum acceptable level.

This matters especially when returns are not normally distributed. In real financial data, returns are often skewed and fat-tailed. Under those conditions, volatility may fail to capture the actual downside risk. Two portfolios can have similar volatility but very different left-tail risk.

PMPT replaces total risk with downside-focused measures.

### Downside Deviation

$$
\text{Downside Deviation}(r^*) =
\sqrt{
\frac{1}{T} \sum_{t=1}^{T} \min(r_t - r^*, 0)^2
}
$$

where:

- $r_t$ is the return of the portfolio at time $t$,
- $r^*$ is the target or minimum acceptable return (often set to 0 or the risk-free rate),
- $T$ is the total number of observations (time periods),
- $\min(r_t - r^*, 0)$ ensures that only **negative deviations** (returns below the target) are counted.

---

### Sortino Ratio

$$
\text{Sortino} =
\frac{\mathbb{E}[r_p - r_f]}{\text{Downside Deviation}(r^*)}
$$

where:

- $r_p$ is the portfolio return,
- $r_f$ is the risk-free rate,
- $\mathbb{E}[r_p - r_f]$ is the expected excess return of the portfolio,
- the denominator is the downside deviation defined above.

Unlike the Sharpe ratio, this measure only penalizes **downside volatility**, not total volatility.

---

### Conditional Value at Risk (CVaR)

If Value at Risk (VaR) at level $\alpha$ is the loss threshold exceeded only $(1-\alpha)\%$ of the time, then:

$$
\text{CVaR}_\alpha = \mathbb{E}[L \mid L \geq \text{VaR}_\alpha]
$$

where:

- $L$ represents portfolio losses (typically $L = -r$),
- $\alpha$ is the confidence level (e.g., $0.95$),
- $\text{VaR}_\alpha$ is the loss threshold such that only the worst $(1-\alpha)\%$ of losses exceed it,
- $\text{CVaR}_\alpha$ is the **average loss in those worst-case scenarios**.

---


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from scipy.optimize import minimize
warnings.filterwarnings('ignore')

prices  = pd.read_csv('data/prices.csv',  index_col=0, parse_dates=True)
returns = pd.read_csv('data/returns.csv', index_col=0, parse_dates=True)

print(f"Loaded {returns.shape[1]} assets × {returns.shape[0]} days")

In [15]:
def min_variance_portfolio(cov):
    n = cov.shape[0]
    w0 = np.ones(n) / n
    res = minimize(
        lambda w: w @ cov @ w,
        w0,
        method="SLSQP",
        bounds=[(0,1)]*n,
        constraints=[{"type":"eq","fun":lambda w: w.sum()-1}]
    )
    return res.x if res.success else w0


def downside_portfolio(window):
    downside = np.minimum(window.values, 0)
    dd = np.sqrt((downside**2).mean(axis=0))
    dd = np.where(dd < 1e-8, 1e-8, dd)
    w = 1/dd
    return w / w.sum()

In [16]:
LOOKBACK = 252
REBAL = 21
MIN_HIST = 252

dates = returns.index[MIN_HIST:]
n = returns.shape[1]

pv = {
    "Equal Weight": [1.0],
    "MPT (Min Var)": [1.0],
    "PMPT (Downside)": [1.0],
}

cw = {k: np.ones(n)/n for k in pv}

for i, date in enumerate(dates):
    idx = returns.index.get_loc(date)

    if i % REBAL == 0:
        window = returns.iloc[idx-LOOKBACK:idx]

        cw["Equal Weight"] = np.ones(n)/n
        cw["MPT (Min Var)"] = min_variance_portfolio(window.cov().values)
        cw["PMPT (Downside)"] = downside_portfolio(window)

    r = returns.iloc[idx].values
    for k in pv:
        pv[k].append(pv[k][-1] * (1 + cw[k] @ r))

portfolio_df = pd.DataFrame(
    pv,
    index=[dates[0]-pd.Timedelta(days=1)] + list(dates)
)

portfolio_df.tail()

,Equal Weight,MPT (Min Var),PMPT (Downside)
2024-12-23,2.787163,2.267806,2.608121
2024-12-24,2.810812,2.287048,2.630243
2024-12-26,2.811398,2.287524,2.630339
2024-12-27,2.793693,2.273118,2.614815
2024-12-30,2.766842,2.251271,2.589804


In [17]:
def sharpe(r):
    return r.mean()*252 / (r.std()*np.sqrt(252))

def sortino(r):
    downside = np.minimum(r, 0)
    dd = np.sqrt((downside**2).mean()) * np.sqrt(252)
    return r.mean()*252 / dd

def cvar(r, alpha=0.95):
    loss = -r
    var = np.quantile(loss, alpha)
    return loss[loss >= var].mean()

metrics = {}

for col in portfolio_df.columns:
    r = portfolio_df[col].pct_change().dropna()

    metrics[col] = {
        "Sharpe": sharpe(r),
        "Sortino": sortino(r),
        "Vol": r.std()*np.sqrt(252),
        "CVaR": cvar(r)
    }

metrics_df = pd.DataFrame(metrics).T
metrics_df.round(3)

,Sharpe,Sortino,Vol,CVaR
Equal Weight,0.957,1.362,0.199,0.029
MPT (Min Var),0.811,1.137,0.190,0.028
PMPT (Downside),0.939,1.335,0.189,0.028


In [18]:
ranks = pd.DataFrame(index=metrics_df.index)

ranks["Sharpe Rank"] = metrics_df["Sharpe"].rank(ascending=False)
ranks["Sortino Rank"] = metrics_df["Sortino"].rank(ascending=False)
ranks["Vol Rank"] = metrics_df["Vol"].rank(ascending=True)
ranks["CVaR Rank"] = metrics_df["CVaR"].rank(ascending=True)

ranks

,Sharpe Rank,Sortino Rank,Vol Rank,CVaR Rank
Equal Weight,1.0,1.0,3.0,3.0
MPT (Min Var),3.0,3.0,2.0,2.0
PMPT (Downside),2.0,2.0,1.0,1.0


## §4 &nbsp; Results

In [ ]:
C = {
    'Equal Weight':    '#888888',
    'MPT (Min Var)':   '#2c5282',
    'PMPT (Downside)': '#c53030',
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.patch.set_facecolor('#f8fafc')

# ── Cumulative performance ──
ax = axes[0]
for k, col in zip(portfolio_df.columns, C):
    ax.plot(portfolio_df.index, portfolio_df[k], label=k, color=C[k], lw=1.8)
ax.set_ylabel('Value ($1 start)')
ax.set_title('Cumulative Performance', fontweight='bold', color='#0f1f3d')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.2, linestyle='--')
ax.set_facecolor('white')

# ── Risk metric comparison (bar chart) ──
ax2 = axes[1]
metric_cols = ['Sharpe', 'Sortino', 'Vol', 'CVaR']
x     = np.arange(len(metric_cols))
width = 0.25
for i, (strat, color) in enumerate(C.items()):
    vals = [metrics_df.loc[strat, m] for m in metric_cols]
    ax2.bar(x + i * width, vals, width, label=strat, color=color, alpha=0.85)
ax2.set_xticks(x + width)
ax2.set_xticklabels(metric_cols)
ax2.set_title('Risk & Performance Metrics', fontweight='bold', color='#0f1f3d')
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.2, linestyle='--')
ax2.set_facecolor('white')

plt.tight_layout()
plt.show()

# ── Metrics table ──
from IPython.display import display
display(metrics_df.round(3).style
    .set_caption('Portfolio Metrics')
    .set_table_styles([{'selector': 'th', 'props': [
        ('background', '#0f1f3d'), ('color', 'white'), ('padding', '6px 12px')
    ]}])
    .highlight_max(subset=['Sharpe', 'Sortino'], color='#dcfce7')
    .highlight_min(subset=['Vol', 'CVaR'],        color='#dbeafe')
)

## §5 Interpretation

The results show that portfolio rankings depend materially on how risk is defined.

Under Sharpe and Sortino (return-adjusted metrics), the Equal Weight portfolio ranks first, followed by the PMPT-based portfolio, while the MPT minimum-variance portfolio performs worst. This reflects the fact that Equal Weight delivers higher average returns, which dominate when performance is evaluated relative to overall volatility.

However, when portfolios are ranked using volatility and CVaR, the ordering reverses. The PMPT (downside-focused) portfolio ranks first, followed by the MPT portfolio, while Equal Weight performs worst due to its higher total and tail risk.

This demonstrates that rankings are not invariant to the choice of risk metric. A portfolio that appears optimal under variance-based evaluation may not remain optimal when downside risk is emphasized. In particular, the PMPT-based approach achieves the lowest volatility and lowest CVaR, indicating superior control of both overall and tail risk.

### Final takeaway
MPT remains a useful framework for portfolio construction, but PMPT provides a more realistic evaluation of performance. The results show that conclusions about “optimal” portfolios depend on whether risk is measured using total volatility or downside-focused metrics.